# Notebook 04 — Baseline Models

**Chapter 3 — Baseline Models for Comparison (Random Forest, SVM, Logistic Regression)**

## Objectives
- Train all three baseline classifiers on the same data used for LSTM
- Evaluate each baseline using standard metrics
- Save trained models and results for Chapter 4 comparison table

---

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src.utils.helpers import set_global_seed, print_banner
from src.config import get_config
from src.utils.paths import PROCESSED_DATA_DIR, BASELINES_DIR
from src.utils.serialization import load_processed_arrays

cfg = get_config()
set_global_seed(cfg.seed)
print_banner('Notebook 04 — Baseline Models')

## 1. Load Processed Arrays

In [ ]:
import json
from src.utils.paths import FINAL_MODEL_DIR
from src.utils.constants import METADATA_JSON

X_train, X_val, X_test, y_train, y_val, y_test = \
    load_processed_arrays(PROCESSED_DATA_DIR)

with open(FINAL_MODEL_DIR / METADATA_JSON) as f:
    metadata = json.load(f)
class_names = metadata['class_names']
n_classes   = metadata['n_classes']

print(f'X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}   |  y_test  : {y_test.shape}')
print(f'Classes : {n_classes} — {class_names}')

## 2. Build Baseline Models

In [ ]:
from src.models.baseline_models import (
    build_random_forest, build_svm, build_logistic_regression
)

rf_model  = build_random_forest(n_estimators=100, class_weight='balanced')
svm_model = build_svm(kernel='rbf', C=1.0, class_weight='balanced')
lr_model  = build_logistic_regression(max_iter=1000)

print('All baseline models built.')

## 3. Train Baselines

In [ ]:
from src.models.baseline_models import train_baseline
import time

models_to_train = [
    ('Random Forest',       rf_model),
    ('SVM',                 svm_model),
    ('Logistic Regression', lr_model),
]

trained_models = {}
training_times = {}

for name, model in models_to_train:
    print(f'Training {name}...', end=' ', flush=True)
    t0 = time.perf_counter()
    trained = train_baseline(model, X_train, y_train, name)
    elapsed = time.perf_counter() - t0
    trained_models[name] = trained
    training_times[name] = elapsed
    print(f'done in {elapsed:.1f}s')

## 4. Evaluate on Test Set

In [ ]:
from src.models.baseline_models import predict_baseline
from src.evaluation.metrics import compute_metrics

all_metrics = {}
for name, model in trained_models.items():
    y_pred, y_prob = predict_baseline(model, X_test, name)
    metrics = compute_metrics(
        y_test, y_pred, y_prob,
        class_names=class_names,
        dataset='nsl_kdd',
        model_name=name,
    )
    all_metrics[name] = metrics

# Summary table
rows = []
for name, m in all_metrics.items():
    rows.append({
        'Model':          name,
        'Accuracy':       m['accuracy'],
        'Precision(M)':   m['precision_macro'],
        'Recall(M)':      m['recall_macro'],
        'F1(Macro)':      m['f1_macro'],
        'ROC-AUC':        m.get('roc_auc', 'N/A'),
        'Train Time (s)': round(training_times.get(name, 0), 1),
    })

results_df = pd.DataFrame(rows)
print('\nBaseline Model Results:')
print(results_df.to_string(index=False))

## 5. Per-Class Metrics

In [ ]:
best_model_name = max(all_metrics, key=lambda k: all_metrics[k]['f1_macro'])
best_per_class  = all_metrics[best_model_name]['per_class_metrics']

print(f'Per-class metrics for best baseline ({best_model_name}):')
print(f'  {"Class":<12}  {"Precision":>10}  {"Recall":>10}  {"F1":>10}  {"Support":>10}')
print('  ' + '-' * 58)
for cls_name, m in best_per_class.items():
    print(f'  {cls_name:<12}  {m["precision"]:>10.4f}  {m["recall"]:>10.4f}  '
          f'{m["f1_score"]:>10.4f}  {m["support"]:>10,}')

## 6. Confusion Matrices

In [ ]:
from src.evaluation.confusion_matrix import plot_confusion_matrix_comparison

cm_data = {}
for name, model in trained_models.items():
    y_pred, _ = predict_baseline(model, X_test, name)
    cm_data[name] = {'y_true': y_test, 'y_pred': y_pred}

path = plot_confusion_matrix_comparison(
    cm_data, dataset='nsl_kdd', class_names=class_names
)
print(f'Confusion matrices saved: {path}')

from IPython.display import Image
Image(str(path))

## 7. Save Baseline Models and Results

In [ ]:
from src.models.baseline_models import save_all_baselines
from src.utils.serialization import save_baseline_results

# Map back to snake_case keys for saving
fitted = {
    'random_forest':       trained_models['Random Forest'],
    'svm':                 trained_models['SVM'],
    'logistic_regression': trained_models['Logistic Regression'],
}
save_all_baselines(fitted, BASELINES_DIR)

# Save metrics
results_flat = {k: {mk: mv for mk, mv in m.items()
                    if mk not in ('confusion_matrix', 'per_class_metrics')}
                for k, m in all_metrics.items()}
save_baseline_results(results_flat, BASELINES_DIR)
print(f'Baseline models and results saved to: {BASELINES_DIR}')

---
**Summary:** Three baseline models trained and evaluated.  
Results saved to `models/baselines/` for Chapter 4 comparison.  
**Next:** Run `05_lstm_training.ipynb`